# 計算技術指標的值

In [14]:
import pandas as pd
import numpy as np
import os

path = os.path.join(os.path.abspath(os.path.dirname(os.getcwd())), 'history_data', 'tw')
print(path)


df = pd.read_csv(path + "/twii_history.csv", encoding='utf-8')

# 計算移動均線 (SMA)
df['MA5'] = df['收盤指數'].rolling(window=5).mean()
df['MA10'] = df['收盤指數'].rolling(window=10).mean()
df['MA20'] = df['收盤指數'].rolling(window=20).mean()

# 計算指數移動平均線 (EMA)
df['EMA12'] = df['收盤指數'].ewm(span=12, adjust=False).mean()
df['EMA26'] = df['收盤指數'].ewm(span=26, adjust=False).mean()

# 計算MACD和DIF
df['DIF'] = df['EMA12'] - df['EMA26']
df['MACD'] = df['DIF'].ewm(span=9, adjust=False).mean()


# 計算RSI
def calculate_rsi(series, period=14):
    delta = series.diff(1)
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

df['RSI'] = calculate_rsi(df['收盤指數'])

# 計算KDJ
low_list = df['最低指數'].rolling(9, min_periods=9).min()
low_list.fillna(value=df['最低指數'].expanding().min(), inplace=True)
high_list = df['最高指數'].rolling(9, min_periods=9).max()
high_list.fillna(value=df['最高指數'].expanding().max(), inplace=True)
rsv = (df['收盤指數'] - low_list) / (high_list - low_list) * 100


# 初始值設定
df['K'] = 50
df['D'] = 50

for i in range(1, len(df)):
    df.loc[df.index[i], 'K'] = (2/3) * df.loc[df.index[i-1], 'K'] + (1/3) * rsv[i]
    df.loc[df.index[i], 'D'] = (2/3) * df.loc[df.index[i-1], 'D'] + (1/3) * df.loc[df.index[i], 'K']


# 顯示DataFrame
df.to_csv(path + "/twii_tech.csv", encoding='utf-8', index=False)


/Users/yanyifu/Documents/_Coding/LLM Predict Stock/history_data/tw


/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_19723/3598096695.py:49: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '55.240163383485836' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[df.index[i], 'K'] = (2/3) * df.loc[df.index[i-1], 'K'] + (1/3) * rsv[i]
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_19723/3598096695.py:50: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '51.74672112782861' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[df.index[i], 'D'] = (2/3) * df.loc[df.index[i-1], 'D'] + (1/3) * df.loc[df.index[i], 'K']
